# Обогащение трамвайных линий

Пайплайн читает девять маршрутов из `data/tram_routes_3221.geojson` и для каждой линии собирает внешние данные. Ключи API не нужны.

| Слой | Источник | Что считается на линии |
| --- | --- | --- |
| Погода | [Open-Meteo](https://api.open-meteo.com/v1/forecast) | температура, ветер, осадки и код погоды в центроиде линии |
| Праздники | [xmlcalendar.ru](https://xmlcalendar.ru/), производственный календарь РФ | праздничные дни `*` за 2024–2026 и ближайший праздник |
| Аварии | [Карта ДТП](https://dtp-stat.ru/media/opendata/moskva.geojson.zip), первоисточник ГИБДД | ДТП 2024–2026 в 150 м от линии |
| Ремонты | OpenStreetMap через Overpass, объекты `construction` | ремонты и стройки в 150 м |
| Пробки | те же магистрали OSM (`motorway`, `trunk`, `primary`) плюс плотность ДТП | балл 0–10: насколько линия идёт вдоль крупных дорог и аварийных мест. Это не живой балл Яндекса: открытой ленты пробок без ключа нет |
| ЧС | опасные коды погоды на линии и лента предупреждений [ГУ МЧС по Москве](https://moscow.mchs.gov.ru/deyatelnost/press-centr/operativnaya-informaciya/vnimanie) | погодное предупреждение на линии и число городских сводок |
| Точки интереса | [Wikipedia geosearch](https://ru.wikipedia.org/w/api.php) | уникальные статьи в 700 м от линии |

Результат пишется в `data/enrichment/`. Архив ДТП кэшируется в `data/enrichment/cache/` и в git не входит.


In [ ]:
from __future__ import annotations

import csv
import json
import math
import time
import zipfile
from collections import defaultdict
from datetime import date, datetime
from pathlib import Path
from urllib.parse import urlencode

import requests

ROOT = Path.cwd()
if not (ROOT / "data" / "tram_routes_3221.geojson").exists():
    ROOT = ROOT.parent

ROUTES_PATH = ROOT / "data" / "tram_routes_3221.geojson"
OUT_DIR = ROOT / "data" / "enrichment"
CACHE_DIR = OUT_DIR / "cache"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_STEP_M = 250
POI_STEP_M = 1200
NEAR_M = 150
POI_RADIUS_M = 700
YEARS = (2024, 2025, 2026)
TODAY = date.today()
HEADERS = {"User-Agent": "mostransport-enrichment/1.0 (tram route study)"}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

HOLIDAY_NAMES = {
    (1, 1): "Новый год",
    (1, 2): "Новогодние каникулы",
    (1, 7): "Рождество Христово",
    (2, 23): "День защитника Отечества",
    (3, 8): "Международный женский день",
    (5, 1): "Праздник Весны и Труда",
    (5, 9): "День Победы",
    (6, 12): "День России",
    (11, 4): "День народного единства",
}

WEATHER_TEXT = {
    0: "ясно", 1: "преимущественно ясно", 2: "переменная облачность", 3: "пасмурно",
    45: "туман", 48: "туман с изморозью",
    51: "морось", 53: "морось", 55: "сильная морось",
    61: "дождь", 63: "дождь", 65: "сильный дождь",
    71: "снег", 73: "снег", 75: "сильный снег",
    80: "ливень", 81: "ливень", 82: "сильный ливень",
    85: "снежный заряд", 86: "сильный снежный заряд",
    95: "гроза", 96: "гроза с градом", 99: "сильная гроза с градом",
}
ALERT_CODES = {45, 48, 65, 67, 75, 82, 86, 95, 96, 99}

print("Корень проекта:", ROOT)


In [ ]:
def fetch_bytes(url, cache_name, timeout=120):
    path = CACHE_DIR / cache_name
    if path.exists() and path.stat().st_size > 0:
        print(f"кэш {cache_name}")
        return path.read_bytes()
    response = SESSION.get(url, timeout=timeout)
    response.raise_for_status()
    path.write_bytes(response.content)
    print(f"скачано {cache_name}: {len(response.content)} байт")
    return response.content


def fetch_json(url, cache_name, timeout=60):
    path = CACHE_DIR / cache_name
    if path.exists() and path.stat().st_size > 0:
        return json.loads(path.read_text(encoding="utf-8"))
    response = SESSION.get(url, timeout=timeout)
    response.raise_for_status()
    path.write_text(response.text, encoding="utf-8")
    return response.json()


def haversine_m(lat1, lon1, lat2, lon2):
    rad = math.radians
    dlat = rad(lat2 - lat1)
    dlon = rad(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(rad(lat1)) * math.cos(rad(lat2)) * math.sin(dlon / 2) ** 2
    return 6371000 * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def lines_of(geometry):
    kind = geometry.get("type")
    coords = geometry.get("coordinates") or []
    if kind == "LineString":
        yield coords
    elif kind == "MultiLineString":
        yield from coords


def route_metrics(geometry, step_m):
    samples = []
    length = 0.0
    for line in lines_of(geometry):
        if len(line) < 2:
            continue
        samples.append((line[0][1], line[0][0]))
        carry = 0.0
        for index in range(1, len(line)):
            lon1, lat1 = line[index - 1]
            lon2, lat2 = line[index]
            dist = haversine_m(lat1, lon1, lat2, lon2)
            length += dist
            if dist == 0:
                continue
            walked = step_m - carry
            while walked <= dist:
                share = walked / dist
                samples.append((lat1 + (lat2 - lat1) * share, lon1 + (lon2 - lon1) * share))
                walked += step_m
            carry = dist - (walked - step_m)
        samples.append((line[-1][1], line[-1][0]))
    return samples, length / 1000


def thinned(samples, step_from, step_to):
    stride = max(1, round(step_to / step_from))
    picked = samples[::stride]
    if samples and picked[-1] != samples[-1]:
        picked.append(samples[-1])
    return picked


def cell_key(lat, lon, size=0.002):
    return (int(lat / size), int(lon / size))


def index_points(items):
    buckets = defaultdict(list)
    for item in items:
        buckets[cell_key(item["lat"], item["lon"])].append(item)
    return buckets


def count_near(samples, buckets, radius_m, identity):
    found = {}
    for lat, lon in samples:
        base_i, base_j = cell_key(lat, lon)
        for di in (-1, 0, 1):
            for dj in (-1, 0, 1):
                for item in buckets.get((base_i + di, base_j + dj), ()):
                    if haversine_m(lat, lon, item["lat"], item["lon"]) <= radius_m:
                        found[identity(item)] = item
    return found


## Линии


In [ ]:
payload = json.loads(ROUTES_PATH.read_text(encoding="utf-8"))
routes = []
for feature in payload["features"]:
    props = feature.get("properties") or {}
    samples, length_km = route_metrics(feature["geometry"], SAMPLE_STEP_M)
    lat = sum(point[0] for point in samples) / len(samples)
    lon = sum(point[1] for point in samples) / len(samples)
    routes.append({
        "number": str(props.get("RouteNumber") or ""),
        "name": props.get("RouteName") or "",
        "samples": samples,
        "poi_samples": thinned(samples, SAMPLE_STEP_M, POI_STEP_M),
        "length_km": round(length_km, 2),
        "lat": lat,
        "lon": lon,
    })
routes.sort(key=lambda route: (len(route["number"]), route["number"]))
for route in routes:
    print(f"{route['number']:>3}  {route['length_km']:6.2f} км  точек {len(route['samples']):4}  {route['name']}")


## Праздники 2024–2026

Берутся дни со звёздочкой в производственном календаре: это официальные праздники, а не обычные субботы и воскресенья.


In [ ]:
def holiday_name(day):
    for shift in range(0, 4):
        for sign in (0, -1, 1):
            probe = date.fromordinal(day.toordinal() + sign * shift)
            title = HOLIDAY_NAMES.get((probe.month, probe.day))
            if title and (shift == 0 or sign):
                return title if shift == 0 else f"{title}, перенос"
    return "Праздничный день"


def load_holidays(year):
    data = fetch_json(
        f"https://xmlcalendar.ru/data/ru/{year}/calendar.json",
        f"calendar_{year}.json",
    )
    marked = []
    for month in data["months"]:
        for token in month["days"].split(","):
            token = token.strip()
            if not token.endswith("*"):
                continue
            day = date(year, month["month"], int(token[:-1]))
            marked.append({"date": day.isoformat(), "name": holiday_name(day)})
    return marked


holidays = []
for year in YEARS:
    holidays.extend(load_holidays(year))
holidays.sort(key=lambda item: item["date"])
holiday_dates = {item["date"] for item in holidays}
by_year = {year: sum(item["date"].startswith(str(year)) for item in holidays) for year in YEARS}
upcoming = [item for item in holidays if item["date"] >= TODAY.isoformat()]
next_holiday = upcoming[0] if upcoming else None
print("Праздников:", by_year)
print("Сегодня праздник:" , TODAY.isoformat() in holiday_dates)
print("Ближайший:", next_holiday)

with (OUT_DIR / "holidays_2024_2026.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["date", "name"])
    writer.writeheader()
    writer.writerows(holidays)


## Погода

Один запрос Open-Meteo на центроиды всех линий. Коды сильного дождя, снега, тумана и грозы помечаются как метеорологическое предупреждение.


In [ ]:
weather_url = "https://api.open-meteo.com/v1/forecast?" + urlencode({
    "latitude": ",".join(f"{route['lat']:.5f}" for route in routes),
    "longitude": ",".join(f"{route['lon']:.5f}" for route in routes),
    "current": "temperature_2m,precipitation,wind_speed_10m,weather_code",
    "timezone": "Europe/Moscow",
})
weather_payload = SESSION.get(weather_url, timeout=40).json()
blocks = weather_payload if isinstance(weather_payload, list) else [weather_payload]
if len(blocks) != len(routes):
    raise RuntimeError(f"Open-Meteo вернул {len(blocks)} блоков на {len(routes)} линий")

for route, block in zip(routes, blocks):
    current = block["current"]
    code = int(current["weather_code"])
    route["weather"] = {
        "time": current["time"],
        "temperature_c": current["temperature_2m"],
        "wind_kmh": current["wind_speed_10m"],
        "precipitation_mm": current["precipitation"],
        "code": code,
        "text": WEATHER_TEXT.get(code, f"код {code}"),
        "alert": code in ALERT_CODES,
    }
    print(f"{route['number']:>3}  {route['weather']['temperature_c']}°  {route['weather']['text']}  ветер {route['weather']['wind_kmh']} км/ч")


## Аварии

Региональная выгрузка Москвы с карты ДТП за январь 2015 — январь 2026. В счёт линии попадают только события 2024–2026 годов не дальше 150 м от трассы. Координаты в источнике иногда записаны как широта/долгота, а не как обычный GeoJSON, поэтому порядок осей проверяется.


In [ ]:
def as_lat_lon(first, second):
    if abs(first) > 50:
        return first, second
    return second, first


def accident_items(raw):
    records = raw["features"] if isinstance(raw, dict) and raw.get("type") == "FeatureCollection" else raw
    for record in records:
        props = record.get("properties") or record
        when = str(props.get("datetime") or props.get("DATE_TIME") or props.get("date") or "")
        if when[:4] not in {str(year) for year in YEARS}:
            continue
        point = (props.get("POINT") or {}).get("coordinates")
        geometry = (record.get("geometry") or {}).get("coordinates")
        pair = point or geometry
        if pair and len(pair) >= 2:
            lat, lon = as_lat_lon(pair[0], pair[1])
        elif props.get("LAT") is not None and props.get("LNG") is not None:
            lat, lon = float(props["LAT"]), float(props["LNG"])
        else:
            continue
        if not (55.4 <= lat <= 56.1 and 37.1 <= lon <= 38.0):
            continue
        text_bits = [props.get("category"), props.get("address"), props.get("tags"), props.get("EM_TYPE"), props.get("STREET")]
        for vehicle in props.get("vehicles") or []:
            text_bits.extend((vehicle.get("category"), vehicle.get("brand")))
        blob = " ".join(str(bit or "") for bit in text_bits).lower()
        yield {
            "id": props.get("id") or props.get("gibdd_number") or props.get("EM_NUMBER") or f"{when}|{lat:.5f}|{lon:.5f}",
            "lat": lat,
            "lon": lon,
            "tram": "трамв" in blob,
        }


archive = fetch_bytes("https://dtp-stat.ru/media/opendata/moskva.geojson.zip", "moskva.geojson.zip")
with zipfile.ZipFile(CACHE_DIR / "moskva.geojson.zip") as bundle:
    name = next(item for item in bundle.namelist() if item.endswith(".geojson") or item.endswith(".json"))
    accidents_raw = json.loads(bundle.read(name))
accidents = list(accident_items(accidents_raw))
accident_index = index_points(accidents)
print(f"ДТП 2024–2026 внутри МКАД-рамки: {len(accidents)}")

for route in routes:
    nearby = count_near(route["samples"], accident_index, NEAR_M, lambda item: item["id"])
    route["accidents"] = len(nearby)
    route["tram_accidents"] = sum(item["tram"] for item in nearby.values())
    route["accidents_per_km"] = round(route["accidents"] / route["length_km"], 2) if route["length_km"] else 0
    print(f"{route['number']:>3}  ДТП {route['accidents']:4}  на км {route['accidents_per_km']:5.2f}  с упоминанием трамвая {route['tram_accidents']}")


## Ремонты и пробки

Overpass отдаёт строящиеся объекты и магистрали в прямоугольнике каждой линии. Ремонт считается, если центр объекта ближе 150 м к трассе. Балл пробок — это открытый индекс риска затора, а не показание датчиков: до 6 баллов за плотность магистралей и до 4 за плотность ДТП.


In [ ]:
OVERPASS_ENDPOINTS = (
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
)


def route_bbox(samples, pad=0.004):
    lats = [point[0] for point in samples]
    lons = [point[1] for point in samples]
    return min(lats) - pad, min(lons) - pad, max(lats) + pad, max(lons) + pad


def element_point(element):
    if "lat" in element and "lon" in element:
        return element["lat"], element["lon"]
    center = element.get("center") or {}
    if "lat" in center and "lon" in center:
        return center["lat"], center["lon"]
    return None


def overpass_for_route(route):
    cache_path = CACHE_DIR / f"overpass_{route['number']}.json"
    if cache_path.exists() and cache_path.stat().st_size > 2:
        return json.loads(cache_path.read_text(encoding="utf-8")), "cache"
    south, west, north, east = route_bbox(route["samples"])
    query = f"""
[out:json][timeout:25];
(
  way["highway"="construction"]({south},{west},{north},{east});
  way["construction"]({south},{west},{north},{east});
  way["highway"~"^(motorway|trunk|primary)$"]({south},{west},{north},{east});
);
out center tags;
"""
    errors = []
    for endpoint in OVERPASS_ENDPOINTS:
        try:
            response = SESSION.post(endpoint, data=query.encode("utf-8"), timeout=30, headers={
                **HEADERS,
                "Content-Type": "text/plain",
            })
            if response.status_code == 200 and response.text.lstrip().startswith("{"):
                cache_path.write_text(response.text, encoding="utf-8")
                return response.json(), endpoint.split("/")[2]
            errors.append(f"{endpoint.split('/')[2]} -> {response.status_code}")
        except requests.RequestException as error:
            errors.append(f"{endpoint.split('/')[2]} -> {error.__class__.__name__}")
        time.sleep(1)
    return None, "; ".join(errors)


road_query_status = "ok"
for route in routes:
    payload, status = overpass_for_route(route)
    repairs, roads = [], []
    if payload is None:
        road_query_status = status
        route["repairs"] = None
        route["major_roads"] = None
        route["traffic_score"] = round(min(10, route["accidents_per_km"] / 2), 1)
        print(f"{route['number']:>3}  overpass недоступен ({status}); пробки только по ДТП: {route['traffic_score']}")
        continue
    for element in payload.get("elements", []):
        point = element_point(element)
        if point is None:
            continue
        tags = element.get("tags") or {}
        item = {"id": f"{element.get('type')}/{element.get('id')}", "lat": point[0], "lon": point[1], "highway": tags.get("highway") or ""}
        if item["highway"] in {"motorway", "trunk", "primary"}:
            roads.append(item)
        else:
            repairs.append(item)
    route["repairs"] = len(count_near(route["samples"], index_points(repairs), NEAR_M, lambda item: item["id"]))
    route["major_roads"] = len(count_near(route["samples"], index_points(roads), NEAR_M, lambda item: item["id"]))
    road_score = min(6, route["major_roads"] / max(route["length_km"], 0.5))
    accident_score = min(4, route["accidents_per_km"] / 3)
    route["traffic_score"] = round(road_score + accident_score, 1)
    print(f"{route['number']:>3}  {status}  ремонты {route['repairs']:3}  магистрали {route['major_roads']:3}  пробки {route['traffic_score']}")
    if status != "cache":
        time.sleep(2)


## ЧС

На линию ставится погодное предупреждение. Городские сводки ГУ МЧС общие для сети: у сайта нет координат события, поэтому число свежих заголовков одинаковое у всех линий, а сами заголовки сохраняются отдельно.


In [ ]:
import re

def mchs_bulletins():
    url = "https://moscow.mchs.gov.ru/deyatelnost/press-centr/operativnaya-informaciya/vnimanie"
    try:
        response = SESSION.get(url, timeout=25)
        response.raise_for_status()
    except requests.RequestException as error:
        return [], f"не загрузилось: {error.__class__.__name__}"
    titles = re.findall(r"<a[^>]+href=\"(/deyatelnost/press-centr/[^\"]+)\"[^>]*>(.*?)</a>", response.text, flags=re.I | re.S)
    cleaned = []
    seen = set()
    for href, raw in titles:
        title = re.sub(r"<[^>]+>", "", raw)
        title = re.sub(r"\s+", " ", title).strip()
        if len(title) < 25 or title in seen:
            continue
        seen.add(title)
        cleaned.append({"title": title, "url": "https://moscow.mchs.gov.ru" + href})
    return cleaned[:15], "ok"


bulletins, bulletin_status = mchs_bulletins()
print("Сводки МЧС:", len(bulletins), bulletin_status)
for item in bulletins[:5]:
    print("-", item["title"])
(OUT_DIR / "emergencies.json").write_text(json.dumps({
    "source": "https://moscow.mchs.gov.ru/deyatelnost/press-centr/operativnaya-informaciya/vnimanie",
    "status": bulletin_status,
    "items": bulletins,
}, ensure_ascii=False, indent=2), encoding="utf-8")


## Точки интереса

Для точек вдоль линии, примерно через каждый километр, запрашиваются статьи Википедии в радиусе 700 м. В счёт линии попадает уникальная статья, даже если она нашлась у нескольких точек. Улицы, переулки и шоссе из выдачи убираются, остаются места: станции, парки, храмы, учреждения.


In [ ]:
def wiki_nearby(lat, lon):
    cache_path = CACHE_DIR / f"wiki_{lat:.3f}_{lon:.3f}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))
    delay = 1.0
    for attempt in range(5):
        response = SESSION.get("https://ru.wikipedia.org/w/api.php", params={
            "action": "query",
            "list": "geosearch",
            "gscoord": f"{lat:.5f}|{lon:.5f}",
            "gsradius": POI_RADIUS_M,
            "gslimit": 30,
            "format": "json",
        }, timeout=30)
        if response.status_code == 429:
            time.sleep(delay)
            delay *= 2
            continue
        response.raise_for_status()
        articles = response.json().get("query", {}).get("geosearch", [])
        cache_path.write_text(json.dumps(articles, ensure_ascii=False), encoding="utf-8")
        time.sleep(0.4)
        return articles
    return []


for route in routes:
    found = {}
    street_marks = ("улица", "переулок", "проезд", "проспект", "шоссе", "тупик", "набережная", "бульвар", "аллея")
    for lat, lon in route["poi_samples"]:
        for article in wiki_nearby(lat, lon):
            title = article["title"]
            if any(mark in title.lower() for mark in street_marks):
                continue
            slot = found.setdefault(article["pageid"], {"title": title, "hits": 0})
            slot["hits"] += 1
    ordered = sorted(found.values(), key=lambda item: (-item["hits"], item["title"]))
    route["poi_count"] = len(ordered)
    route["poi_examples"] = [item["title"] for item in ordered[:8]]
    print(f"{route['number']:>3}  POI {route['poi_count']:4}  {', '.join(route['poi_examples'][:3])}")


## Сводка по линиям


In [ ]:
rows = []
for route in routes:
    weather = route["weather"]
    rows.append({
        "route_number": route["number"],
        "route_name": route["name"],
        "length_km": route["length_km"],
        "temperature_c": weather["temperature_c"],
        "wind_kmh": weather["wind_kmh"],
        "precipitation_mm": weather["precipitation_mm"],
        "weather": weather["text"],
        "weather_alert": int(weather["alert"]),
        "holiday_today": int(TODAY.isoformat() in holiday_dates),
        "next_holiday": "" if next_holiday is None else f"{next_holiday['date']} {next_holiday['name']}",
        "holidays_2024": by_year.get(2024, 0),
        "holidays_2025": by_year.get(2025, 0),
        "holidays_2026": by_year.get(2026, 0),
        "accidents_2024_2026": route["accidents"],
        "accidents_per_km": route["accidents_per_km"],
        "tram_related_accidents": route["tram_accidents"],
        "repairs_nearby": route["repairs"],
        "major_roads_nearby": route["major_roads"],
        "traffic_score": route["traffic_score"],
        "emergency_weather": int(weather["alert"]),
        "emergency_bulletins": len(bulletins),
        "poi_count": route["poi_count"],
        "poi_examples": "; ".join(route["poi_examples"]),
    })

fieldnames = list(rows[0])
with (OUT_DIR / "routes_enriched.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows([{key: "" if value is None else value for key, value in row.items()} for row in rows])

(OUT_DIR / "routes_enriched.json").write_text(json.dumps({
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "near_m": NEAR_M,
    "poi_radius_m": POI_RADIUS_M,
    "overpass_status": road_query_status,
    "mchs_status": bulletin_status,
    "traffic_score": "до 6 баллов за магистрали OSM на километр и до 4 за ДТП на километр; без живой ленты пробок",
    "routes": rows,
}, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Строк: {len(rows)}")
print(f"CSV:   {OUT_DIR / 'routes_enriched.csv'}")
print(f"JSON:  {OUT_DIR / 'routes_enriched.json'}")
print()
header = f"{'№':>4} {'км':>6} {'погода':<22} {'ДТП':>5} {'рем.':>5} {'пробки':>6} {'ЧС':>3} {'POI':>5}"
print(header)
for row in rows:
    repairs = "-" if row["repairs_nearby"] is None else f"{row['repairs_nearby']:>5}"
    print(
        f"{row['route_number']:>4} {row['length_km']:6.2f} {row['weather']:<22} "
        f"{row['accidents_2024_2026']:5} {repairs:>5} {row['traffic_score']:6} "
        f"{row['emergency_weather']:3} {row['poi_count']:5}"
    )
